In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Libraries and Data Cleaning


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load data
train_df = pd.read_csv("/content/drive/MyDrive/Machine Learning/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Machine Learning/test.csv")

# Basic text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)  # remove punctuation
    text = re.sub("\s+", " ", text).strip()  # normalize whitespace
    return text

# Apply cleaning
train_df['text_clean'] = train_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

#LogReg Model and Output

In [ ]:
# Split train/validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text_clean'], train_df['target'], test_size=0.2, random_state=42)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Validate
val_preds = model.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

# Predict on test data
test_vec = tfidf.transform(test_df['text_clean'])
test_preds = model.predict(test_vec)

# Prepare submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})
submission.to_csv("/content/drive/MyDrive/Machine Learning/disaster_tweet_submission.csv", index=False)
print("Submission file saved to /content/drive/MyDrive/Machine Learning/disaster_tweet_submission.csv")



Validation Accuracy: 0.8108995403808273
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       874
           1       0.83      0.70      0.76       649

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523

Submission file saved to /content/drive/MyDrive/Machine Learning/disaster_tweet_submission.csv
